# Vision A threshold baseline check (Phase 2, EXP-1)

EXP-1 gave a core Dice of 0.11 against the final infarct, with a median of 0.03. This notebook checks whether that comes from a bug or from the data. For each development case it measures:
- how much of the infarct lies inside Tmax > 6 s, and on the side the baseline calls affected;
- the rCBF inside the infarct;
- Dice and volume at several rCBF cut-offs.

It also counts how many occlusions nearest an MCA label overlap it. The baseline itself is unchanged, at 30%.

**Before you start:** CPU runtime (Runtime → Change runtime type → None).

**Run it:** Runtime → Run all. Allow about 5 minutes, which is an estimate; EXP-1 scored at 0.21 s per case.

**Send back:** the `OK:` line. Results are saved as `runs/EXP-1_thresholds_baseline/diagnostics.json`.

## 1. Settings

In [ ]:
ROOT = '/content/drive/MyDrive/StrokeCopilot'
SLIM_TAR = f'{ROOT}/slim/isles24_slim_v1.tar'
SLIM_SHA256 = 'b0afd16e4cbfb16a69fb5190083fc3869f5dcc6c92f86f10135b0de1f59f4728'
FOLDS_SHA256 = '023813f2dba930b7521653f1549b1fa1369b8bd29b63658175c5cea58410d291'
RUN_DIR = f'{ROOT}/runs/EXP-1_thresholds_baseline'
LOCAL_DIR = '/content/isles24'
EXPECTED_DEVELOPMENT = 129
REPO_REF = 'main'

## 2. Install the package and mount Drive

In [ ]:
import hashlib, json, os, subprocess, sys, time
from google.colab import drive

repo = 'https://github.com/parsa-hke/strokecopilot.git'
found = subprocess.run(['git', 'ls-remote', repo, REPO_REF], capture_output=True, text=True)
if found.returncode or not found.stdout.strip():
    raise RuntimeError(('Could not read the repo. ' + found.stderr)[-800:])
COMMIT = found.stdout.split()[0]
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'strokecopilot[isles] @ git+{repo}@{COMMIT}'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError(result.stderr[-800:])
del repo, found, result
print(f'Package installed at commit {COMMIT[:7]}.')
drive.mount('/content/drive')

from strokecopilot.data import archive_listing as al
from strokecopilot.data import isles24, splits
from strokecopilot.vision.perfusion import thresholds, volumes

if hashlib.sha256(open(splits.FOLDS_PATH, 'rb').read()).hexdigest() != FOLDS_SHA256:
    raise ValueError('The installed split is not the fixed one. Stop and check it.')
folds = splits.load_folds()
if not os.path.exists(os.path.join(RUN_DIR, 'metrics.json')):
    raise FileNotFoundError('EXP-1 has not finished; run 11_visionA_thresholds first.')
if os.path.exists(os.path.join(RUN_DIR, 'diagnostics.json')):
    raise FileExistsError('diagnostics.json already exists. Nothing was changed.')
started = time.time()
if al.sha256_of(SLIM_TAR) != SLIM_SHA256:
    raise ValueError('The tarball on Drive is not the one the build recorded. Stop and check it.')
slim_root = isles24.fetch_slim(SLIM_TAR, parent=LOCAL_DIR)
fold_of = {case: int(k) for k, ids in folds['folds'].items() for case in ids}
development = isles24.list_cases(slim_root)
if len(development) != EXPECTED_DEVELOPMENT or set(development) != set(fold_of):
    raise ValueError(f'The loader lists {len(development)} cases, not the {EXPECTED_DEVELOPMENT} development cases.')
print(f'Data ready in {time.time() - started:.0f} s.')

## 3. Measure every development case

In [ ]:
rows = []
started = time.time()
for i, case in enumerate(development, 1):
    loaded = isles24.load_case(case, slim_root, names=('ncct', 'cbf', 'tmax', 'occlusion', 'cow', 'infarct'))
    row = {'fold': fold_of[case], 'ref_ml': loaded.native_volumes['infarct_ml'], **thresholds.diagnose_case(loaded)}
    row.update({f'occlusion_{k}': v for k, v in volumes.occlusion_evidence(loaded.arrays['occlusion'], loaded.arrays['cow'], loaded.spacing_mm).items()})
    rows.append(row)
    if i % 20 == 0 or i == len(development):
        print(f'{i}/{len(development)} cases, {time.time() - started:.0f} s', flush=True)

## 4. Summary

In [ ]:
import numpy as np

def pct(values, points=(5, 25, 50, 75, 95)):
    values = [v for v in values if v is not None]
    return {f'p{p}': round(float(np.percentile(values, p)), 3) for p in points} if values else None

by_size = sorted(rows, key=lambda r: r['ref_ml'])
tertiles = [by_size[i * len(by_size) // 3:(i + 1) * len(by_size) // 3] for i in range(3)]
cuts = thresholds.DIAGNOSTIC_RCBF
diagnostics = {
    'experiment': 'EXP-1', 'git_commit': COMMIT, 'cases': len(rows),
    'infarct_in_tmax6_share': pct([r['infarct_in_tmax6_share'] for r in rows]),
    'infarct_on_affected_side_share': pct([r['infarct_on_affected_side_share'] for r in rows]),
    'cases_mostly_on_other_side': sum((r['infarct_on_affected_side_share'] or 1) < 0.5 for r in rows),
    'tmax6_dice': pct([r['tmax6_dice'] for r in rows]),
    'infarct_rcbf_p50': pct([r.get('infarct_rcbf_p50') for r in rows]),
    'by_rcbf_cut': {f'{c:g}': {'dice': pct([r.get(f'dice_rcbf{c:g}') for r in rows], (25, 50, 75)),
                              'dice_mean': round(float(np.mean([r[f'dice_rcbf{c:g}'] for r in rows if r.get(f'dice_rcbf{c:g}') is not None])), 3),
                              'error_ml': pct([r[f'core_ml_rcbf{c:g}'] - r['ref_ml'] for r in rows if f'core_ml_rcbf{c:g}' in r], (5, 50, 95))} for c in cuts},
    'by_infarct_tertile': [{'ref_ml': pct([r['ref_ml'] for r in t], (0, 50, 100)), 'dice_rcbf0.3': pct([r.get('dice_rcbf0.3') for r in t], (25, 50, 75)),
                            'infarct_in_tmax6_share': pct([r['infarct_in_tmax6_share'] for r in t], (25, 50, 75)),
                            'core_ml_rcbf0.3': pct([r.get('core_ml_rcbf0.3') for r in t], (25, 50, 75))} for t in tertiles],
    'mca_nearest': {
        'cases': sum(r['occlusion_site'] == 'MCA' for r in rows),
        'overlapping_label': sum(r['occlusion_site'] == 'MCA' and r['occlusion_distance_mm'] == 0 for r in rows),
        'mostly_inside_label': sum(r['occlusion_site'] == 'MCA' and r['occlusion_outside_share'] < 0.5 for r in rows),
        'within_1.5_mm': sum(r['occlusion_site'] == 'MCA' and r['occlusion_distance_mm'] <= 1.5 for r in rows),
    },
}
print(json.dumps(diagnostics, indent=1))

## 5. Save and check

In [ ]:
text = json.dumps(diagnostics, indent=1)
if any(case in text for case in folds['heldout']):
    raise ValueError('A held-out ID reached the output. Nothing was saved.')
saved = os.path.join(RUN_DIR, 'diagnostics.json')
with open(saved, 'w') as f:
    f.write(text)
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)
if open(saved).read() != text:
    raise ValueError('diagnostics.json read back from Drive differs. Run this section again.')
drive.flush_and_unmount()
d = diagnostics
print(f"OK: {d['cases']} cases; infarct in Tmax>6 median {d['infarct_in_tmax6_share']['p50']}; on affected side median {d['infarct_on_affected_side_share']['p50']}; "
      f"infarct rCBF median {d['infarct_rcbf_p50']['p50']}; MCA overlapping {d['mca_nearest']['overlapping_label']} of {d['mca_nearest']['cases']}; read back.")